In [1]:
!pip install requests


Defaulting to user installation because normal site-packages is not writeable


In [7]:
import requests
import json
import datetime
import csv

# Function to create a bounding box (bbox) from a single lat/lon point and a distance in degrees
def create_bbox_from_point(lat, lon, distance_deg=0.1):
    # Generate a bounding box around the given point
    # distance_deg is the half width/height of the bounding box
    lat_min = lat - distance_deg
    lat_max = lat + distance_deg
    lon_min = lon - distance_deg
    lon_max = lon + distance_deg
    return f"{lon_min},{lat_min},{lon_max},{lat_max}"

# Function to identify the Sentinel satellite based on the spacecraft_id
def identify_satellite(satellite_code):
    satellite_mapping = {
        'S2A': 'Sentinel-2A',
        'S2B': 'Sentinel-2B',
        # You can add more satellites as necessary
    }
    
    return satellite_mapping.get(satellite_code, "Unknown Sentinel Satellite")

# Function to fetch Sentinel-2 data
def get_sentinel_data_from_stac(bbox, start_datetime, end_datetime):
    sentinel_url = "https://catalogue.dataspace.copernicus.eu/stac/collections/SENTINEL-2/items"
    params = {
        'bbox': bbox,
        'datetime': f"{start_datetime}/{end_datetime}"
    }
    
    response = requests.get(sentinel_url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        sentinel_data = []
        for feature in data.get('features', []):
            satellite_code = feature["properties"]["platformSerialIdentifier"]
            satellite_name = "Sentinel 2"+ satellite_code
            datetime = feature["properties"]["start_datetime"]
            sentinel_data.append((datetime, satellite_name))
        return sentinel_data
    else:
        print(f"Error fetching Sentinel data: {response.status_code} - {response.text}")
        return []

# Function to fetch Landsat data
def get_landsat_data_from_stac(bbox, start_datetime, end_datetime):
    landsat_url = "https://landsatlook.usgs.gov/stac-server/search"
    params = {
        'bbox': bbox,
        'datetime': f"{start_datetime}/{end_datetime}"
    }
    
    response = requests.get(landsat_url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        landsat_data = []
        for feature in data.get('features', []):
            sat_code = feature["properties"]["platform"]
            sat_name = sat_code
            datetime = feature["properties"]["datetime"]
            landsat_data.append((datetime, sat_name))
        return landsat_data
    else:
        print(f"Error fetching Landsat data: {response.status_code} - {response.text}")
        return []

today = datetime.date.today()
# Function to write data to CSV
def write_to_csv(sentinel_datetimes, landsat_datetimes, lat, lon, filename=f"Outputs/satellite_data_{today}.csv"):
    header = ['Date', 'Satellite', 'Lat (DEG)', 'Lon (DEG)']
    
    # Open CSV file for writing
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)  # Write header

        # Write Sentinel-2 data
        for datetime in sentinel_datetimes:
            writer.writerow([datetime[0], datetime[1], lat, lon])

        # Write Landsat data
        for datetime in landsat_datetimes:
            writer.writerow([datetime[0], datetime[1], lat, lon])

            
# Example usage
# Input point: latitude and longitude
lat = 47.81306  # Latitude of the point
lon = 13.04667  # Longitude of the point

# Create bbox around the point (using a distance of 0.1 degrees for simplicity)
bbox = create_bbox_from_point(lat, lon, distance_deg=0.5)
start_datetime = "2025-02-01T00:00:00Z"  # Start datetime
end_datetime = "2025-06-13T23:59:59Z"  # End datetime

# Fetching Sentinel-2 data
sentinel_datetimes = get_sentinel_data_from_stac(bbox, start_datetime, end_datetime)
print("Sentinel-2 capture datetimes:", sentinel_datetimes)

# Fetching Landsat data
landsat_datetimes = get_landsat_data_from_stac(bbox, start_datetime, end_datetime)
print("Landsat capture datetimes:", landsat_datetimes)

def remove_duplicates(data):
    # Using a set to keep track of unique (datetime, satellite) pairs
    unique_data = set(data)
    return list(unique_data)
print("removed duplicates")

sentinel_unique_datetimes = remove_duplicates(sentinel_datetimes);
landsat_unique_datetimes = remove_duplicates(landsat_datetimes);

# Write the captured data to a CSV file
write_to_csv(sentinel_unique_datetimes, landsat_unique_datetimes, lat, lon)
print("Data saved to satellite_data.csv")

Sentinel-2 capture datetimes: [('2025-02-01T10:13:01.025000Z', 'Sentinel 2C'), ('2025-02-01T10:13:01.025000Z', 'Sentinel 2C'), ('2025-02-01T10:13:01.025000Z', 'Sentinel 2C'), ('2025-02-01T10:13:01.025000Z', 'Sentinel 2C'), ('2025-02-01T10:13:01.025000Z', 'Sentinel 2C'), ('2025-02-01T10:13:01.025000Z', 'Sentinel 2C'), ('2025-02-01T10:13:01.025000Z', 'Sentinel 2C'), ('2025-02-01T10:13:01.025000Z', 'Sentinel 2C'), ('2025-02-03T10:01:29.024000Z', 'Sentinel 2B'), ('2025-02-03T10:01:29.024000Z', 'Sentinel 2B'), ('2025-02-03T10:01:29.024000Z', 'Sentinel 2B'), ('2025-02-06T10:11:09.025000Z', 'Sentinel 2B'), ('2025-04-01T10:00:41.024000Z', 'Sentinel 2A'), ('2025-04-01T10:00:41.024000Z', 'Sentinel 2A'), ('2025-04-01T10:00:41.024000Z', 'Sentinel 2A'), ('2025-04-02T10:10:51.025000Z', 'Sentinel 2C'), ('2025-04-02T10:10:51.025000Z', 'Sentinel 2C'), ('2025-04-02T10:10:51.025000Z', 'Sentinel 2C'), ('2025-04-02T10:10:51.025000Z', 'Sentinel 2C'), ('2025-04-02T10:10:51.025000Z', 'Sentinel 2C')]
Landsat c

PermissionError: [Errno 13] Permission denied: 'Outputs/satellite_data_2025-04-16.csv'

In [21]:
import requests
import datetime
import csv

# Create bounding box from lat/lon point
def create_bbox_from_point(lat, lon, distance_deg=0.1):
    lat_min = lat - distance_deg
    lat_max = lat + distance_deg
    lon_min = lon - distance_deg
    lon_max = lon + distance_deg
    return [lon_min, lat_min, lon_max, lat_max]

# Map satellite platform code to name
def identify_satellite(satellite_code):
    satellite_mapping = {
        'sentinel-2a': 'Sentinel-2A',
        'sentinel-2b': 'Sentinel-2B',
        'sentinel-2c': 'Sentinel-2C',
    }
    return satellite_mapping.get(satellite_code, f"Unknown Sentinel Satellite")

# Fetch Sentinel-2 L1C + L2A data
def get_sentinel_data_from_stac(bbox, start_datetime, end_datetime):
    url = "https://stac.dataspace.copernicus.eu/v1/search"
    payload = {
        "collections": ["sentinel-2-l1c", "sentinel-2-l2a"],
        "bbox": bbox,
        "datetime": f"{start_datetime}/{end_datetime}",
        "limit": 100
    }
    headers = {
        "Content-Type": "application/json"
    }
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        data = response.json()
        results = []
        for feature in data.get("features", []):
            platform_code = feature["properties"].get("platform", "unknown").lower()
            satellite_name = identify_satellite(platform_code)
            capture_time = feature["properties"]["datetime"]
            results.append((capture_time, satellite_name))
        return results
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return []

# Fetch Landsat data
def get_landsat_data_from_stac(bbox, start_datetime, end_datetime):
    url = "https://landsatlook.usgs.gov/stac-server/search"
    payload = {
        "collections": ["landsat-c2l2-sr"],
        "bbox": bbox,
        "datetime": f"{start_datetime}/{end_datetime}",
        "limit": 100
    }
    headers = {
        "Content-Type": "application/json"
    }
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        data = response.json()
        results = []
        for feature in data.get("features", []):
            platform = feature["properties"].get("platform", "Unknown")
            capture_time = feature["properties"]["datetime"]
            results.append((capture_time, platform))
        return results
    else:
        print(f"Error fetching Landsat data: {response.status_code} - {response.text}")
        return []

# Remove duplicates
def remove_duplicates(data):
    return list(set(data))

# Write data to CSV
def write_to_csv(sentinel_data, landsat_data, lat, lon, filename):
    header = ['Date', 'Satellite', 'Lat (DEG)', 'Lon (DEG)']
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        for dt in sentinel_data:
            writer.writerow([dt[0], dt[1], lat, lon])
        for dt in landsat_data:
            writer.writerow([dt[0], dt[1], lat, lon])

# ----------- MAIN -----------

lat = 47.81306
lon = 13.04667
bbox = create_bbox_from_point(lat, lon, distance_deg=0.25)

start_datetime = "2025-02-01T00:00:00Z"
end_datetime = "2025-06-13T23:59:59Z"

sentinel_data = get_sentinel_data_from_stac(bbox, start_datetime, end_datetime)
landsat_data = get_landsat_data_from_stac(bbox, start_datetime, end_datetime)

print("Sentinel-2 captures:", sentinel_data)
print("Landsat captures:", landsat_data)

sentinel_unique = remove_duplicates(sentinel_data)
landsat_unique = remove_duplicates(landsat_data)

today = datetime.date.today()
filename = f"Outputs/satellite_data_{today}.csv"
write_to_csv(sentinel_unique, landsat_unique, lat, lon, filename)

print(f"Data saved to {filename}")


Sentinel-2 captures: [('2025-04-14T10:07:01.024Z', 'Sentinel-2A'), ('2025-04-14T10:07:01.024Z', 'Sentinel-2A'), ('2025-04-14T10:07:01.024Z', 'Sentinel-2A'), ('2025-04-14T10:07:01.024Z', 'Sentinel-2A'), ('2025-04-14T10:07:01.024Z', 'Sentinel-2A'), ('2025-04-14T10:07:01.024Z', 'Sentinel-2A'), ('2025-04-14T10:07:01.024Z', 'Sentinel-2A'), ('2025-04-14T10:07:01.024Z', 'Sentinel-2A'), ('2025-04-14T10:00:29.024Z', 'Sentinel-2B'), ('2025-04-14T10:00:29.024Z', 'Sentinel-2B'), ('2025-04-14T10:00:29.024Z', 'Sentinel-2B'), ('2025-04-14T10:00:29.024Z', 'Sentinel-2B'), ('2025-04-14T10:00:29.024Z', 'Sentinel-2B'), ('2025-04-14T10:00:29.024Z', 'Sentinel-2B'), ('2025-04-11T10:00:41.024Z', 'Sentinel-2A'), ('2025-04-11T10:00:41.024Z', 'Sentinel-2A'), ('2025-04-11T10:00:41.024Z', 'Sentinel-2A'), ('2025-04-11T10:00:41.024Z', 'Sentinel-2A'), ('2025-04-07T10:05:59.024Z', 'Sentinel-2B'), ('2025-04-07T10:05:59.024Z', 'Sentinel-2B'), ('2025-04-07T10:05:59.024Z', 'Sentinel-2B'), ('2025-04-07T10:05:59.024Z', 'Sen